# Dataset Creation - Part A
מחברת זו מבצעת: חישת נתונים, בדיקות איכות, הכנה ויצירת משתנה המטרה RainTomorrow.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
RAW_PATH = Path("data/raw/sensed_data.csv")
PROCESSED_PATH = Path("data/processed/final_dataset.csv")

In [ ]:
df = pd.read_csv(RAW_PATH)
df.head()

## Data Quality Checks

In [ ]:
quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_ratio": df.isna().mean().round(4),
    "dtype": df.dtypes.astype(str)
}).sort_values("missing_ratio", ascending=False)
quality.head(20)

## Distributions (at least 7 numeric variables)

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=["number"]).columns]
selected_cols = numeric_cols[:7]
fig, axes = plt.subplots(len(selected_cols), 1, figsize=(8, 4 * len(selected_cols)))
if len(selected_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, selected_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()

## Target Creation: RainTomorrow
לפי ההוראות: בתוך כל location, לאחר מיון לפי date, RainTomorrow = 1 אם precipitation_sum של היום הבא ≥ 1.0 מ"מ, אחרת 0.
היום האחרון בכל מיקום נמחק (אין לו "מחר"). מידע מהיום הבא לא משמש כמאפיין.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["location", "date"]).reset_index(drop=True)

next_precip = df.groupby("location")["precipitation_sum"].shift(-1)
next_date = df.groupby("location")["date"].shift(-1)

# Sanity check: "next row" must really be the next calendar day
gaps = df[next_date.notna() & ((next_date - df["date"]).dt.days != 1)]
print("date gaps inside a location:", len(gaps))

df["RainTomorrow"] = (next_precip >= 1.0).astype(int)
df = df[next_precip.notna()].reset_index(drop=True)  # drop last day per location
df[["location", "date", "precipitation_sum", "RainTomorrow"]].head(10)

In [ ]:
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(exclude=["number"]).columns:
    mode = df[col].mode(dropna=True)
    if len(mode) > 0:
        df[col] = df[col].fillna(mode.iloc[0])
df.isna().sum().sum()

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved: {PROCESSED_PATH}")